# Problema 4 — El Rezago Educativo como Deuda Sistémica
## ODS 4 (Educación de Calidad) × ODS 16 (Paz, Justicia e Instituciones)

> México casi universalizó la alfabetización... pero eso oculta tres décadas de deuda acumulada
> en calidad docente, infraestructura digital, acceso superior y un contexto institucional que
> frena el aprendizaje antes de que empiece.

---
| Acto | Ángulo |
|------|--------|
| I | La ilusión del éxito — tres velocidades en el aprendizaje |
| II | La brecha digital: escuelas sin conexión |
| III | El docente desarmado — formación y multigrado |
| IV | La violencia como techo de cristal |
| V | La institución desacreditada — corrupción y desconfianza |
| VI | El nudo sistémico — cómo ODS 16 frena a ODS 4 |
| VII | Radar final — ¿dónde estamos parados? |

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

# ── Datos ─────────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/consolidated/indicadores_largo.csv')

def get_serie(ind_id):
    return df[df['indicator_id'] == ind_id].sort_values('periodo').copy()

# ── Paleta ODS 4 × ODS 16 ─────────────────────────────────────────────────────
C_ODS4   = '#C5192D'   # rojo ODS 4
C_ODS16  = '#00689D'   # azul ODS 16
C_MEJORA = '#2DC653'
C_ALERTA = '#E63946'
C_ACENTO = '#F4A261'
C_FONDO  = '#F8F9FA'
C_LINEA  = '#343A40'

# ── Normalizador 0-100 (50 = baseline, 100 = meta) ───────────────────────────
def normalizar(actual, base, meta, mayor_mejor):
    if mayor_mejor:
        prog = (actual - base) / (meta - base) if (meta - base) != 0 else 0
    else:
        prog = (base - actual) / (base - meta) if (base - meta) != 0 else 0
    return round(max(0, min(100, 50 + 50 * prog)), 1)

print('Setup completo. Indicadores ODS 4 y ODS 16 disponibles.')

Setup completo. Indicadores ODS 4 y ODS 16 disponibles.


## Acto I — La Ilusión del Éxito
La tasa de alfabetización llega al **99.2 %** en 2024 — una cifra que México repite con orgullo.
Pero detrás del titular se esconden dos crisis silenciosas: la matrícula de educación superior
apenas alcanza el **43.8 %** (vs. ~65 % promedio OCDE) y solo el **37 %** de adultos participa
en formación continua. México sabe leer; no sabe aprender permanentemente.

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 1 — La Ilusión del Éxito Educativo
# ══════════════════════════════════════════════════════════════════════════════
alfab = get_serie('4.6.1.A')
hed   = get_serie('4R.3.1')
adult = get_serie('4.3.1')

fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Tasa de Alfabetizacion (%)',
        'Matricula Educacion Superior (%)',
        'Adultos en Formacion Continua (%)'
    ],
    horizontal_spacing=0.11
)

# ── Panel 1: Alfabetización ───────────────────────────────────────────────────
fig1.add_trace(go.Scatter(
    x=alfab['periodo'], y=alfab['valor'],
    mode='lines',
    line=dict(color=C_ODS4, width=3),
    fill='tozeroy', fillcolor='rgba(197,25,45,0.08)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Alfabetizacion: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig1.add_annotation(
    x=2000, y=96.8,
    text='<b>96.8%</b><br>2000',
    showarrow=False, font=dict(size=9, color=C_ODS4), row=1, col=1
)
fig1.add_annotation(
    x=2024, y=99.2,
    text='<b>99.2%</b><br>2024',
    showarrow=True, arrowhead=2, ax=30, ay=-25,
    font=dict(size=11, color=C_ODS4),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ODS4, borderwidth=1.5,
    row=1, col=1
)

# ── Panel 2: Educación superior ───────────────────────────────────────────────
hed_colors = [C_ACENTO if v < 40 else C_MEJORA for v in hed['valor']]
fig1.add_trace(go.Bar(
    x=hed['periodo'], y=hed['valor'],
    marker_color=hed_colors,
    marker_line_color='white', marker_line_width=0.5,
    text=['{:.1f}%'.format(v) for v in hed['valor']],
    textposition='outside', textfont=dict(size=10),
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Matricula superior: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=2)

fig1.add_hline(
    y=65, line_width=1.5, line_dash='dash', line_color='#6C757D',
    annotation_text='Promedio OCDE ~65%',
    annotation_position='right',
    annotation_font=dict(size=9, color='#6C757D'),
    row=1, col=2
)
fig1.add_annotation(
    x=2023, y=43.83,
    text='<b>43.8%</b><br>21 pp abajo<br>de OCDE',
    showarrow=True, arrowhead=2, ax=0, ay=40,
    font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5,
    row=1, col=2
)

# ── Panel 3: Adultos en formación continua ────────────────────────────────────
adult_colors = [C_ALERTA if v < 35.5 else C_ACENTO if v < 37 else C_ODS16 for v in adult['valor']]
fig1.add_trace(go.Scatter(
    x=adult['periodo'], y=adult['valor'],
    mode='lines+markers',
    line=dict(color=C_ODS16, width=3),
    marker=dict(size=9, color=adult_colors, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(0,104,157,0.07)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Adultos en formacion: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=3)

fig1.add_annotation(
    x=2020, y=36.71,
    text='COVID:<br>retroceso<br>en formacion',
    showarrow=True, arrowhead=2, ax=45, ay=-30,
    font=dict(size=9, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ALERTA, borderwidth=1,
    row=1, col=3
)
fig1.add_annotation(
    x=2024, y=37.07,
    text='<b>37.1%</b><br>sin recuperacion<br>post-COVID',
    showarrow=False,
    font=dict(size=10, color=C_ODS16),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ODS16, borderwidth=1,
    row=1, col=3
)

fig1.update_yaxes(title_text='% poblacion', row=1, col=1)
fig1.update_yaxes(title_text='% matricula bruta', row=1, col=2)
fig1.update_yaxes(title_text='% adultos (ult. 12 meses)', row=1, col=3)
fig1.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig1.update_yaxes(showgrid=True, gridcolor='#E9ECEF')

fig1.update_layout(
    title=dict(
        text='<b>La Ilusion del Exito Educativo — Tres Velocidades en el Aprendizaje</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Mexico casi universalizo la alfabetizacion, pero se rezago 21 pp en ed. superior vs OCDE</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    margin=dict(t=100, b=60)
)
fig1.show()

## Acto II — La Brecha Digital en el Salón de Clases
Mientras la electricidad llega al **95.6 %** de las escuelas, el internet sólo alcanza al
**46.7 %** y los equipos de cómputo en funcionamiento al **47.6 %** — y ese último número
*empeoró* desde 2018 (era 54.3 %). La pandemia aceleró la necesidad de conectividad
pero las escuelas llegaron sin red.

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 2 — La Brecha Digital en las Escuelas
# ══════════════════════════════════════════════════════════════════════════════
elec = get_serie('4.A.1.A')
inet = get_serie('4.A.1.B')
comp = get_serie('4.A.1.C')

fig2 = go.Figure()

# Área sombreada de crisis digital (por debajo del 60%)
fig2.add_shape(
    type='rect', x0=2017.6, x1=2024.4, y0=0, y1=60,
    fillcolor='rgba(230,57,70,0.04)', line_width=0, layer='below'
)

fig2.add_trace(go.Scatter(
    x=elec['periodo'], y=elec['valor'],
    mode='lines+markers',
    name='Electricidad',
    line=dict(color=C_MEJORA, width=3),
    marker=dict(size=10),
    hovertemplate='<b>%{x}</b><br>Con electricidad: <b>%{y:.1f}%</b><extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=comp['periodo'], y=comp['valor'],
    mode='lines+markers',
    name='Computadoras en funcionamiento',
    line=dict(color=C_ACENTO, width=3, dash='dash'),
    marker=dict(size=10, symbol='square'),
    hovertemplate='<b>%{x}</b><br>Con computadoras: <b>%{y:.1f}%</b><extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=inet['periodo'], y=inet['valor'],
    mode='lines+markers',
    name='Acceso a Internet',
    line=dict(color=C_ALERTA, width=3, dash='dot'),
    marker=dict(size=10, symbol='diamond'),
    hovertemplate='<b>%{x}</b><br>Con internet: <b>%{y:.1f}%</b><extra></extra>'
))

fig2.add_annotation(
    x=2018, y=55.5,
    text='2018: 54.3%<br>con computadoras',
    showarrow=True, arrowhead=2, ax=-60, ay=-25,
    font=dict(size=9, color=C_ACENTO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ACENTO, borderwidth=1
)
fig2.add_annotation(
    x=2021, y=29.55,
    text='<b>COVID 2021:</b><br>internet cae a 29.6%',
    showarrow=True, arrowhead=2, ax=70, ay=30,
    font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5
)
fig2.add_annotation(
    x=2024, y=47.6,
    text='<b>2024:</b> computadoras 47.6%<br>MENOS que en 2018',
    showarrow=True, arrowhead=2, ax=0, ay=-50,
    font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5
)

fig2.update_layout(
    title=dict(
        text='<b>La Brecha Digital en las Escuelas (2018–2024)</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Electricidad universal (95.6%), pero solo 1 de cada 2 escuelas tiene internet o computadoras</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    xaxis_title='Año', yaxis_title='% de escuelas',
    yaxis=dict(range=[0, 105], showgrid=True, gridcolor='#E9ECEF'),
    xaxis=dict(showgrid=True, gridcolor='#E9ECEF'),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.22, xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.8)', bordercolor='#DEE2E6', borderwidth=1
    ),
    margin=dict(t=110, b=90)
)
fig2.show()

## Acto III — El Docente Desarmado
**1 de cada 5 maestros de preescolar** no cuenta con la certificación mínima requerida.
En secundaria —el nivel crítico para retención escolar— el porcentaje sin certifcación es
del **13 %**. Además, el **15 %** de las escuelas del país opera en modelo **multigrado**:
un solo docente para todos los grados, en su mayoría en comunidades rurales e indígenas.

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 3 — El Docente Desarmado
# ══════════════════════════════════════════════════════════════════════════════
prof_pre  = get_serie('4.C.1.A')
prof_pri  = get_serie('4.C.1.B')
prof_sec  = get_serie('4.C.1.C')
multigrado = get_serie('4N.1.1')

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Docentes con Formacion Minima Certificada (%)',
        'Escuelas Multigrado (% del total)'
    ],
    horizontal_spacing=0.13
)

# ── Panel 1: Formación docente ────────────────────────────────────────────────
fig3.add_trace(go.Scatter(
    x=prof_pre['periodo'], y=prof_pre['valor'],
    mode='lines+markers', name='Preescolar',
    line=dict(color=C_ALERTA, width=3),
    marker=dict(size=8),
    hovertemplate='<b>%{x}</b><br>Preescolar: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig3.add_trace(go.Scatter(
    x=prof_sec['periodo'], y=prof_sec['valor'],
    mode='lines+markers', name='Secundaria',
    line=dict(color=C_ACENTO, width=3, dash='dash'),
    marker=dict(size=8, symbol='square'),
    hovertemplate='<b>%{x}</b><br>Secundaria: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig3.add_trace(go.Scatter(
    x=prof_pri['periodo'], y=prof_pri['valor'],
    mode='lines+markers', name='Primaria',
    line=dict(color=C_MEJORA, width=3, dash='dot'),
    marker=dict(size=8, symbol='diamond'),
    hovertemplate='<b>%{x}</b><br>Primaria: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig3.add_hline(
    y=100, line_width=1, line_dash='dash', line_color='#ADB5BD',
    annotation_text='Meta: 100%',
    annotation_position='right',
    annotation_font=dict(size=9, color='#6C757D'),
    row=1, col=1
)

fig3.add_annotation(
    x=2024, y=80.33,
    text='<b>80.3%</b> preescolar<br>1 de cada 5 maestros<br>sin certificacion',
    showarrow=True, arrowhead=2, ax=-85, ay=0,
    font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5,
    row=1, col=1
)

# ── Panel 2: Multigrado ───────────────────────────────────────────────────────
mg_colors = [C_ALERTA if v > 16 else C_ACENTO if v > 15 else C_ODS4 for v in multigrado['valor']]
fig3.add_trace(go.Bar(
    x=multigrado['periodo'], y=multigrado['valor'],
    marker_color=mg_colors,
    marker_line_color='white', marker_line_width=0.5,
    name='Multigrado',
    text=['{:.1f}%'.format(v) for v in multigrado['valor']],
    textposition='outside', textfont=dict(size=10),
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Escuelas multigrado: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=2)

fig3.add_annotation(
    x=2020, y=16.8,
    text='15-16% de escuelas:<br>1 docente para<br>todos los grados',
    showarrow=False,
    font=dict(size=10, color=C_LINEA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor='#ADB5BD', borderwidth=1,
    row=1, col=2
)

fig3.update_yaxes(title_text='% con formacion certificada', row=1, col=1)
fig3.update_yaxes(title_text='% del total de escuelas', row=1, col=2)
fig3.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig3.update_yaxes(showgrid=True, gridcolor='#E9ECEF')

fig3.update_layout(
    title=dict(
        text='<b>El Docente Desarmado — Formacion Docente y Escuelas Multigrado</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             '1 de cada 5 maestros de preescolar no tiene certificacion minima; 15% de escuelas son multigrado</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.22, xanchor='center', x=0.35,
        bgcolor='rgba(255,255,255,0.8)', bordercolor='#DEE2E6', borderwidth=1
    ),
    margin=dict(t=100, b=85)
)
fig3.show()

## Acto IV — La Violencia como Techo de Cristal
¿Cómo aprende un niño cuando su comunidad vive bajo amenaza? La tasa de homicidios en México
(**25.4/100k en 2024**) quintuplica el umbral de epidemia de la OMS (5/100k). El **24 %** de la
población fue víctima de un delito en 2024. Solo el **40 %** se siente seguro al caminar de noche.
El ODS 16 no es sólo una meta de gobernanza — es el suelo sobre el que descansa el ODS 4.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 4 — La Violencia como Techo de Cristal
# ══════════════════════════════════════════════════════════════════════════════
homicidios   = get_serie('16.1.1')
victimizacion = get_serie('16N.1.1')
seg_nocturna  = get_serie('16.1.4')
pretrial      = get_serie('16.3.2')

fig4 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Tasa de Homicidios (por 100,000 hab.)',
        'Victimizacion Delictiva (% poblacion)',
        'Percepcion de Seguridad Nocturna (%)',
        'Detenidos sin Condena (% de reos)'
    ],
    vertical_spacing=0.15, horizontal_spacing=0.12
)

# ── Panel 1: Homicidios ───────────────────────────────────────────────────────
hom_colors = [C_ALERTA if v > 24 else C_ACENTO if v > 17 else C_MEJORA for v in homicidios['valor']]
fig4.add_trace(go.Scatter(
    x=homicidios['periodo'], y=homicidios['valor'],
    mode='lines+markers',
    line=dict(color=C_ODS16, width=3),
    marker=dict(size=9, color=hom_colors, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(0,104,157,0.08)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Homicidios: <b>%{y:.1f}</b>/100k<extra></extra>'
), row=1, col=1)

fig4.add_hline(
    y=5, line_width=1.5, line_dash='dash', line_color=C_MEJORA,
    annotation_text='Limite OMS: 5/100k',
    annotation_position='right',
    annotation_font=dict(size=9, color=C_MEJORA),
    row=1, col=1
)
fig4.add_annotation(
    x=2018, y=29.12,
    text='<b>Pico 2018:</b><br>29.1 — 5.8x OMS',
    showarrow=True, arrowhead=2, ax=-65, ay=-20,
    font=dict(size=9, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5,
    row=1, col=1
)

# ── Panel 2: Victimización ────────────────────────────────────────────────────
vict_colors = [C_ALERTA if v > 27 else C_ACENTO if v > 24 else C_ODS4 for v in victimizacion['valor']]
fig4.add_trace(go.Bar(
    x=victimizacion['periodo'], y=victimizacion['valor'],
    marker_color=vict_colors,
    marker_line_color='white', marker_line_width=0.5,
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Victimizacion: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=2)

fig4.add_annotation(
    x=2024, y=24.13,
    text='<b>1 de cada 4</b><br>mexicanos victima<br>de delito (2024)',
    showarrow=True, arrowhead=2, ax=0, ay=-55,
    font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5,
    row=1, col=2
)

# ── Panel 3: Seguridad nocturna ───────────────────────────────────────────────
seg_colors = [C_ALERTA if v < 40 else C_ACENTO if v < 42 else C_ODS4 for v in seg_nocturna['valor']]
fig4.add_trace(go.Bar(
    x=seg_nocturna['periodo'], y=seg_nocturna['valor'],
    marker_color=seg_colors,
    marker_line_color='white', marker_line_width=0.5,
    text=['{:.1f}%'.format(v) for v in seg_nocturna['valor']],
    textposition='outside', textfont=dict(size=10),
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Se siente seguro: <b>%{y:.1f}%</b><extra></extra>'
), row=2, col=1)

fig4.add_hline(
    y=50, line_width=1, line_dash='dot', line_color='#ADB5BD',
    annotation_text='Minimo aceptable 50%',
    annotation_position='right',
    annotation_font=dict(size=9, color='#6C757D'),
    row=2, col=1
)

# ── Panel 4: Presos sin condena ───────────────────────────────────────────────
pt_colors = [C_ALERTA if v > 40 else C_ACENTO if v > 35 else C_ODS4 for v in pretrial['valor']]
fig4.add_trace(go.Scatter(
    x=pretrial['periodo'], y=pretrial['valor'],
    mode='lines+markers',
    line=dict(color=C_ODS16, width=3),
    marker=dict(size=9, color=pt_colors, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(0,104,157,0.08)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Sin condena: <b>%{y:.1f}%</b><extra></extra>'
), row=2, col=2)

fig4.add_annotation(
    x=2021, y=42.13,
    text='<b>42.1%</b> en 2021<br>4 de cada 10 presos<br>sin sentencia',
    showarrow=True, arrowhead=2, ax=-75, ay=-20,
    font=dict(size=9, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5,
    row=2, col=2
)

fig4.update_yaxes(showgrid=True, gridcolor='#E9ECEF')
fig4.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig4.update_yaxes(title_text='Homicidios / 100k hab', row=1, col=1)
fig4.update_yaxes(title_text='% victimizados', row=1, col=2)
fig4.update_yaxes(title_text='% que se siente seguro', row=2, col=1)
fig4.update_yaxes(title_text='% sin condena', row=2, col=2)

fig4.update_layout(
    title=dict(
        text='<b>La Violencia como Techo de Cristal — ODS 16 en Crisis</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Homicidios 5x el limite OMS, 1 de cada 4 mexicanos victimizado — contexto que bloquea el aprendizaje</span>',
        x=0.5, xanchor='center'
    ),
    height=620, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    margin=dict(t=100, b=60)
)
fig4.show()

## Acto V — La Institución Desacreditada
Solo **6 de cada 10 mexicanos** confía en la policía estatal, el Ministerio Público y los jueces.
En 2023, el **14 %** de ciudadanos pagó al menos un soborno al tener contacto con un funcionario.
Cuando las instituciones no inspiran confianza, el pacto social que hace significativa la educación
se erosiona: ¿para qué estudiar si el sistema no recompensa el mérito?

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 5 — La Institución Desacreditada
# ══════════════════════════════════════════════════════════════════════════════
policia    = get_serie('16N.2.1')
mp_trust   = get_serie('16N.2.2')
jueces     = get_serie('16N.2.4')
soborno_c  = get_serie('16.5.1')
soborno_n  = get_serie('16.5.2')

fig5 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Confianza en Instituciones de Justicia (%)',
        'Tasa de Sobornos — Corrupcion (%)'
    ],
    horizontal_spacing=0.13
)

# ── Panel 1: Confianza institucional ──────────────────────────────────────────
fig5.add_trace(go.Scatter(
    x=policia['periodo'], y=policia['valor'],
    mode='lines+markers', name='Policia Estatal',
    line=dict(color=C_ODS16, width=2.5),
    marker=dict(size=8),
    hovertemplate='<b>%{x}</b><br>Confianza policia: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig5.add_trace(go.Scatter(
    x=mp_trust['periodo'], y=mp_trust['valor'],
    mode='lines+markers', name='MP / Fiscalia',
    line=dict(color=C_ACENTO, width=2.5, dash='dash'),
    marker=dict(size=8, symbol='square'),
    hovertemplate='<b>%{x}</b><br>Confianza MP: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig5.add_trace(go.Scatter(
    x=jueces['periodo'], y=jueces['valor'],
    mode='lines+markers', name='Jueces',
    line=dict(color=C_ODS4, width=2.5, dash='dot'),
    marker=dict(size=8, symbol='diamond'),
    hovertemplate='<b>%{x}</b><br>Confianza jueces: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig5.add_hline(
    y=70, line_width=1, line_dash='dash', line_color='#ADB5BD',
    annotation_text='Umbral de confianza solida (70%)',
    annotation_position='right',
    annotation_font=dict(size=9, color='#6C757D'),
    row=1, col=1
)
fig5.add_annotation(
    x=2025, y=57.05,
    text='<b>2025:</b> MP cae<br>a 57% de confianza',
    showarrow=True, arrowhead=2, ax=-75, ay=20,
    font=dict(size=9, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ALERTA, borderwidth=1,
    row=1, col=1
)

# ── Panel 2: Corrupción ───────────────────────────────────────────────────────
fig5.add_trace(go.Scatter(
    x=soborno_c['periodo'], y=soborno_c['valor'],
    mode='lines+markers', name='Ciudadanos',
    line=dict(color=C_ALERTA, width=3),
    marker=dict(size=10),
    hovertemplate='<b>%{x}</b><br>Soborno ciudadanos: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=2)

fig5.add_trace(go.Scatter(
    x=soborno_n['periodo'], y=soborno_n['valor'],
    mode='lines+markers', name='Negocios',
    line=dict(color=C_ACENTO, width=3, dash='dash'),
    marker=dict(size=10, symbol='square'),
    hovertemplate='<b>%{x}</b><br>Soborno negocios: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=2)

fig5.add_annotation(
    x=2019, y=15.73,
    text='<b>Pico 2019:</b><br>1 de cada 6<br>pago soborno',
    showarrow=True, arrowhead=2, ax=-70, ay=-20,
    font=dict(size=9, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5,
    row=1, col=2
)
fig5.add_annotation(
    x=2023, y=13.97,
    text='<b>2023: 14.0%</b><br>vs meta <2%',
    showarrow=False, xanchor='left',
    font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ALERTA, borderwidth=1.5,
    row=1, col=2
)

fig5.update_yaxes(title_text='% con confianza', row=1, col=1)
fig5.update_yaxes(title_text='% que pago soborno', row=1, col=2)
fig5.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig5.update_yaxes(showgrid=True, gridcolor='#E9ECEF')

fig5.update_layout(
    title=dict(
        text='<b>La Institucion Desacreditada — Confianza y Corrupcion en Mexico</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Solo 6 de cada 10 confian en policia, MP y jueces; 1 de cada 7 ciudadanos pago soborno en 2023</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.22, xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.8)', bordercolor='#DEE2E6', borderwidth=1
    ),
    margin=dict(t=100, b=90)
)
fig5.show()

## Acto VI — El Nudo Sistémico
¿Son independientes el rezago educativo y la crisis institucional? Los datos dicen que **no**.
La correlación de Spearman entre la tasa de homicidios y la disponibilidad de computadoras
en las escuelas es **r = −0.96**: donde más violencia, menos recursos tecnológicos en aulas.
Y la satisfacción con la educación pública sube casi en paralelo con la formación docente (r ≈ 0.94).
ODS 4 y ODS 16 no son metas separadas — son las dos caras de la misma deuda sistémica.

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 6 — El Nudo Sistémico: ODS 4 × ODS 16
# ══════════════════════════════════════════════════════════════════════════════

# ── Scatter 1: Homicidios vs Computadoras ─────────────────────────────────────
hom  = get_serie('16.1.1')
comp = get_serie('4.A.1.C')
m1 = hom[['periodo','valor']].rename(columns={'valor':'homicidios'}).merge(
    comp[['periodo','valor']].rename(columns={'valor':'compus'}), on='periodo'
)
r_hc, p_hc = spearmanr(m1['homicidios'], m1['compus'])

# ── Scatter 2: Formación docente vs Satisfacción educativa ────────────────────
educ_sat = get_serie('16.6.2.A')
prof_sec = get_serie('4.C.1.C')
m2 = educ_sat[['periodo','valor']].rename(columns={'valor':'satisf'}).merge(
    prof_sec[['periodo','valor']].rename(columns={'valor':'formacion'}), on='periodo'
)
r_sf, p_sf = spearmanr(m2['satisf'], m2['formacion'])

fig6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Homicidios vs Computadoras en Escuelas (Spearman r = {:.2f})'.format(r_hc),
        'Formacion Docente vs Satisfaccion Educativa (Spearman r = {:.2f})'.format(r_sf)
    ],
    horizontal_spacing=0.14
)

# Scatter 1
yr_colors_1 = [C_ALERTA if h > 25 else C_ACENTO if h > 20 else C_MEJORA for h in m1['homicidios']]
fig6.add_trace(go.Scatter(
    x=m1['homicidios'], y=m1['compus'],
    mode='markers+text',
    text=[str(int(y)) for y in m1['periodo']],
    textposition='top center',
    textfont=dict(size=9, color=C_LINEA),
    marker=dict(size=12, color=yr_colors_1, line=dict(width=1.5, color='white')),
    showlegend=False,
    hovertemplate='<b>%{text}</b><br>Homicidios: %{x:.1f}/100k<br>Computadoras: %{y:.1f}%<extra></extra>'
), row=1, col=1)

# Trendline scatter 1
m1_sort = m1.sort_values('homicidios')
poly1 = np.polyfit(m1_sort['homicidios'], m1_sort['compus'], 1)
x_trend1 = np.linspace(m1_sort['homicidios'].min(), m1_sort['homicidios'].max(), 50)
fig6.add_trace(go.Scatter(
    x=x_trend1, y=np.polyval(poly1, x_trend1),
    mode='lines', line=dict(color=C_ODS16, width=2, dash='dash'),
    showlegend=False, hoverinfo='skip'
), row=1, col=1)

fig6.add_annotation(
    x=28, y=51,
    text='A mayor violencia,<br>menos tecnologia educativa',
    showarrow=False,
    font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ALERTA, borderwidth=1,
    row=1, col=1
)

# Scatter 2
yr_colors_2 = [C_ODS4 if s > 67 else C_ACENTO for s in m2['satisf']]
fig6.add_trace(go.Scatter(
    x=m2['formacion'], y=m2['satisf'],
    mode='markers+text',
    text=[str(int(y)) for y in m2['periodo']],
    textposition='top center',
    textfont=dict(size=9, color=C_LINEA),
    marker=dict(size=12, color=yr_colors_2, line=dict(width=1.5, color='white')),
    showlegend=False,
    hovertemplate='<b>%{text}</b><br>Formacion docente sec.: %{x:.1f}%<br>Satisfaccion ed.: %{y:.1f}%<extra></extra>'
), row=1, col=2)

# Trendline scatter 2
m2_sort = m2.sort_values('formacion')
poly2 = np.polyfit(m2_sort['formacion'], m2_sort['satisf'], 1)
x_trend2 = np.linspace(m2_sort['formacion'].min(), m2_sort['formacion'].max(), 50)
fig6.add_trace(go.Scatter(
    x=x_trend2, y=np.polyval(poly2, x_trend2),
    mode='lines', line=dict(color=C_ODS4, width=2, dash='dash'),
    showlegend=False, hoverinfo='skip'
), row=1, col=2)

fig6.add_annotation(
    x=86, y=68.5,
    text='Mas docentes certificados<br>mayor satisfaccion ciudadana',
    showarrow=False,
    font=dict(size=10, color=C_ODS4),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ODS4, borderwidth=1,
    row=1, col=2
)

fig6.update_xaxes(showgrid=True, gridcolor='#E9ECEF')
fig6.update_yaxes(showgrid=True, gridcolor='#E9ECEF')
fig6.update_xaxes(title_text='Homicidios / 100,000 hab', row=1, col=1)
fig6.update_yaxes(title_text='Escuelas con computadoras (%)', row=1, col=1)
fig6.update_xaxes(title_text='Docentes secundaria certificados (%)', row=1, col=2)
fig6.update_yaxes(title_text='Satisfaccion con ed. publica (%)', row=1, col=2)

fig6.update_layout(
    title=dict(
        text='<b>El Nudo Sistemico — ODS 4 y ODS 16 no son Metas Independientes</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Donde hay mas violencia hay menos tecnologia educativa; mas docentes certificados elevan la satisfaccion</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    margin=dict(t=100, b=60)
)
fig6.show()

## Acto VII — Radar: ¿Dónde Estamos Parados?
Puntuación normalizada 0-100 donde **50 = línea base histórica**, **100 = meta cumplida**.
La educación avanza en algunos frentes (alfabetización, formación docente) pero hay retrocesos
en infraestructura digital. Las instituciones muestran un perfil de estancamiento en violencia
y corrupción, con modesta mejoría en confianza institucional.

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 7 — Radar ODS 4 × ODS 16
# ══════════════════════════════════════════════════════════════════════════════

# (label, base, actual, meta, mayor_mejor)
ods4_data = [
    ('Alfabetizacion\n(meta 100%)',         95.48, 99.20, 100.0, True),
    ('Formacion docente\nsecundaria',        70.78, 86.84, 100.0, True),
    ('Internet en\nescuelas',               38.29, 46.65,  80.0, True),
    ('Educacion\nsuperior',                 34.51, 43.83,  65.0, True),
    ('Adultos en\nformacion continua',       33.55, 37.07,  60.0, True),
]

ods16_data = [
    ('Homicidios\n(meta <5/100k)',           22.45, 25.36,   5.0, False),
    ('Seguridad\nnocturna',                  38.62, 41.33,  70.0, True),
    ('Confianza\nen policia',                55.50, 61.47,  80.0, True),
    ('Corrupcion\n(meta <2%)',               12.59, 13.97,   2.0, False),
    ('Confianza\nen jueces',                 53.74, 60.53,  80.0, True),
]

cats4  = [d[0] for d in ods4_data]
scores4 = [normalizar(d[2], d[1], d[3], d[4]) for d in ods4_data]

cats16  = [d[0] for d in ods16_data]
scores16 = [normalizar(d[2], d[1], d[3], d[4]) for d in ods16_data]

fig7 = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'polar'}, {'type': 'polar'}]],
    subplot_titles=['ODS 4 — Educacion de Calidad', 'ODS 16 — Paz, Justicia e Instituciones']
)

# ODS 4 radar
theta4 = cats4 + [cats4[0]]
r4     = scores4 + [scores4[0]]

fig7.add_trace(go.Scatterpolar(
    r=r4, theta=theta4,
    fill='toself', fillcolor='rgba(197,25,45,0.18)',
    line=dict(color=C_ODS4, width=2.5),
    name='ODS 4 actual',
    hovertemplate='<b>%{theta}</b><br>Score: <b>%{r:.1f}/100</b><extra></extra>'
), row=1, col=1)

fig7.add_trace(go.Scatterpolar(
    r=[50]*len(cats4) + [50],
    theta=theta4,
    fill='toself', fillcolor='rgba(173,181,189,0.12)',
    line=dict(color='#ADB5BD', width=1.5, dash='dot'),
    name='Linea base (50)',
    showlegend=True
), row=1, col=1)

# ODS 16 radar
theta16 = cats16 + [cats16[0]]
r16     = scores16 + [scores16[0]]

fig7.add_trace(go.Scatterpolar(
    r=r16, theta=theta16,
    fill='toself', fillcolor='rgba(0,104,157,0.18)',
    line=dict(color=C_ODS16, width=2.5),
    name='ODS 16 actual',
    hovertemplate='<b>%{theta}</b><br>Score: <b>%{r:.1f}/100</b><extra></extra>'
), row=1, col=2)

fig7.add_trace(go.Scatterpolar(
    r=[50]*len(cats16) + [50],
    theta=theta16,
    fill='toself', fillcolor='rgba(173,181,189,0.12)',
    line=dict(color='#ADB5BD', width=1.5, dash='dot'),
    name='Linea base (50)',
    showlegend=False
), row=1, col=2)

# Anotaciones de score
for i, (cat, score) in enumerate(zip(cats4, scores4)):
    color = C_MEJORA if score >= 70 else C_ACENTO if score >= 55 else C_ALERTA
    print('{}: {:.1f}/100'.format(cat.replace(chr(10), ' '), score))

print()
for cat, score in zip(cats16, scores16):
    color = C_MEJORA if score >= 70 else C_ACENTO if score >= 55 else C_ALERTA
    print('{}: {:.1f}/100'.format(cat.replace(chr(10), ' '), score))

polar_style = dict(
    radialaxis=dict(
        visible=True, range=[0, 100],
        tickvals=[25, 50, 75, 100],
        ticktext=['25', 'Base', '75', 'Meta'],
        gridcolor='#E9ECEF', linecolor='#DEE2E6'
    ),
    angularaxis=dict(tickfont=dict(size=10))
)

fig7.update_layout(
    polar=polar_style,
    polar2=polar_style,
    title=dict(
        text='<b>Radar ODS 4 x ODS 16 — Progreso desde Linea Base (50) hacia Meta (100)</b><br>'
             '<span style="font-size:12px;color:#666;font-weight:normal">'
             'Educacion avanza en calidad docente; instituciones retroceden en violencia y corrupcion</span>',
        x=0.5, xanchor='center'
    ),
    height=560, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.08, xanchor='center', x=0.5
    ),
    margin=dict(t=110, b=60)
)
fig7.show()

Alfabetizacion (meta 100%): 91.2/100
Formacion docente secundaria: 77.5/100
Internet en escuelas: 60.0/100
Educacion superior: 65.3/100
Adultos en formacion continua: 56.7/100

Homicidios (meta <5/100k): 41.7/100
Seguridad nocturna: 54.3/100
Confianza en policia: 62.2/100
Corrupcion (meta <2%): 43.5/100
Confianza en jueces: 62.9/100


---
## Conclusiones: La Deuda Sistémica

| Indicador | Valor actual | Meta ODS | Brecha |
|-----------|-------------|---------|--------|
| Matrícula educación superior | 43.8% | ~60% | **−16 pp** |
| Docentes preescolar certificados | 80.3% | 100% | **−20 pp** |
| Escuelas con internet | 46.7% | 80% | **−33 pp** |
| Tasa de homicidios | 25.4/100k | <5/100k | **5× el límite** |
| Tasa de soborno ciudadano | 14.0% | <2% | **7× la meta** |
| Confianza en instituciones | ~60% | >80% | **−20 pp** |

> **Mensaje clave**: México no tiene un problema de educación y un problema de seguridad por separado.
> Tiene una *deuda sistémica* donde la violencia erosiona la infraestructura educativa, la corrupción
> desmotiva el esfuerzo escolar y la falta de formación docente reproduce el rezago generación tras generación.
> Resolver el ODS 4 requiere resolver el ODS 16 — y viceversa.